# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
GENRES = ['blues', 'classical', 'country' ,'disco' ,'hiphop', 'jazz', 'metal','pop', 'reggae', 'rock'] # Make the list of all genres available (alphabetical order)
STEMS = {'bass': 'bass.wav', 'drums': 'drums.wav', 'other': 'other.wav', 'vocals': 'vocals.wav'} # Write here stems file name
STEM_KEYS = ['bass', 'drums', 'other', 'vocals']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0 #Enter index as per Q10.

In [4]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s.replace('.wav',''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav',''): [] for s in STEMS} for g in GENRES}

    rng = random.Random(seed)

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        if not os.path.isdir(genre_path):
            continue

        valid_songs = []
        for song in os.listdir(genre_path):
            song_path = os.path.join(genre_path, song)

            if not all(os.path.isfile(os.path.join(song_path, f)) for f in STEMS.values()):
                continue

            valid_songs.append(song_path)

        rng.shuffle(valid_songs)
        split_idx = int(len(valid_songs)*(1-val_split))

        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]

        def add_to_dict(target, songs):
            for s in songs:
                for k,v in STEMS.items():
                    target[genre][k].append(os.path.join(s,v))

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    return train_dataset, val_dataset


tr, val = build_dataset(DATA_ROOT)


In [5]:
class MashupDataset(Dataset):
    def __init__(self, dataset_dict, augment=False):
        self.samples = []
        self.augment = augment

        for genre in GENRES:
            paths = dataset_dict[genre]['vocals']
            for i in range(len(paths)):
                stems_dict = {stem: dataset_dict[genre][stem][i] for stem in STEM_KEYS}
                self.samples.append((stems_dict, genre))

    def __len__(self):
        return len(self.samples)

    def load_audio(self, path):
        y, _ = librosa.load(path, sr=SR)
        return y

    def add_noise(self, y):
        noise = np.random.randn(len(y))
        noise = noise / (np.max(np.abs(noise)) + 1e-6)
        return y + noise * (10 ** (-TARGET_SNR_DB / 20))

    def spec_augment(self, mel):
        t = mel.shape[1]
        t_mask = random.randint(5, 20)
        t0 = random.randint(0, t - t_mask)
        mel[:, t0:t0+t_mask] = 0

        f = mel.shape[0]
        f_mask = random.randint(5, 15)
        f0 = random.randint(0, f - f_mask)
        mel[f0:f0+f_mask, :] = 0

        return mel

    def extract_features(self, y):
        mel = librosa.feature.melspectrogram(
            y=y, sr=SR, n_fft=N_FFT,
            hop_length=HOP_LENGTH, n_mels=N_MELS
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.mean())/(mel_db.std()+1e-6)

        target_len = int(SR * DURATION / HOP_LENGTH)
        if mel_db.shape[1] < target_len:
            mel_db = np.pad(mel_db, ((0,0),(0,target_len-mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :target_len]

        if self.augment:
            mel_db = self.spec_augment(mel_db)

        return mel_db

    def __getitem__(self, idx):
        stems_dict, genre = self.samples[idx]

        stem_choice = random.choice(STEM_KEYS)
        y = self.load_audio(stems_dict[stem_choice])

        if self.augment:
            y = self.add_noise(y)

        mel = self.extract_features(y)

        x = torch.tensor(mel).unsqueeze(0).float()
        y = torch.tensor(GENRES.index(genre)).long()

        return x, y

In [6]:
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )

        self.fc = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(256, len(GENRES))
        )

    def forward(self,x):
        x = self.net(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [7]:
train_dataset = MashupDataset(tr, augment=True)
val_dataset   = MashupDataset(val, augment=False)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [8]:
def train_model(model, train_loader, val_loader, epochs=10):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x,y in train_loader:
            x,y = x.cuda(), y.cuda()

            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out,y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()
        correct,total = 0,0

        with torch.no_grad():
            for x,y in val_loader:
                x,y = x.cuda(), y.cuda()
                pred = torch.argmax(model(x),1)
                correct += (pred==y).sum().item()
                total += y.size(0)

        acc = correct/total
        print(epoch+1, total_loss, acc)

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(),"best_model.pth")

model = CNNModel().cuda()
train_model(model, train_loader, val_loader)

1 116.00243413448334 0.1
2 110.03623843193054 0.08823529411764706
3 111.98855364322662 0.1411764705882353
4 112.08311426639557 0.11764705882352941
5 108.73739337921143 0.10588235294117647
6 106.28506469726562 0.07058823529411765
7 109.60172712802887 0.11764705882352941
8 107.33687448501587 0.11176470588235295
9 107.38850522041321 0.13529411764705881
10 106.8734188079834 0.13529411764705881


In [9]:
def predict_wav(model, file_path):
    model.eval()

    # If the path is a directory, pick the first .wav file inside
    if os.path.isdir(file_path):
        wav_files = [f for f in os.listdir(file_path) if f.endswith(".wav")]
        if not wav_files:
            raise FileNotFoundError(f"No .wav files found in {file_path}")
        file_path = os.path.join(file_path, wav_files[0])

    # Load the audio file
    y, _ = librosa.load(file_path, sr=SR)
    y = y / (np.max(np.abs(y)) + 1e-6)

    # Compute mel spectrogram
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

    # Pad or truncate to fixed length
    target_len = int(SR * DURATION / HOP_LENGTH)
    if mel_db.shape[1] < target_len:
        mel_db = np.pad(mel_db, ((0,0),(0,target_len - mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :target_len]

    # Convert to tensor
    x = torch.tensor(mel_db).unsqueeze(0).unsqueeze(0).float().cuda()

    # Predict
    with torch.no_grad():
        pred = torch.softmax(model(x), dim=1)

    final_class = torch.argmax(pred, dim=1).item()
    return GENRES[final_class]

In [10]:
model.load_state_dict(torch.load("best_model.pth"))

genre = GENRE_TO_TEST
song_file = os.listdir(os.path.join(DATA_ROOT, genre))[SONG_INDEX]
song_path = os.path.join(DATA_ROOT, genre, song_file)

print(song_file)          # e.g., song0001.wav
print("Song path:", song_path)
print("Is directory?", os.path.isdir(song_path))

print("Actual:", genre)
print("Predicted:", predict_wav(model, song_path))

rock.00052
Song path: /kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/rock/rock.00052
Is directory? True
Actual: rock
Predicted: pop


In [11]:
TEST_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups'

test_files = [
    os.path.join(TEST_ROOT, f)
    for f in os.listdir(TEST_ROOT)
    if f.endswith(".wav")
]

print("Total test files:", len(test_files))

Total test files: 3020


In [12]:
submission_records = []

for file_path in tqdm(test_files):
    song_id = str(os.path.basename(file_path)).replace("song","").replace(".wav","")

    pred = predict_wav(model, file_path)

    submission_records.append({
        "id": song_id,
        "genre": pred
    })

100%|██████████| 3020/3020 [03:28<00:00, 14.51it/s]


In [13]:
submission_df = pd.DataFrame(submission_records)
submission_df.to_csv("/kaggle/working/submission.csv", index=False)
submission_df.head()

,id,genre
0,2501,pop
1,0956,pop
2,2874,pop
3,1970,pop
4,0718,metal


In [14]:
# def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
#     records = []

#     total_files = sum(len(paths) for genre in dataset_dict.values() for paths in genre.values())
#     print(f"Total files to check: {total_files}")

#     sample_rates = set()

#     for genre, stems in dataset_dict.items():
#         for stem_name, file_paths in stems.items():
#             for file_path in file_paths:
#                 try:
#                     # Load audio
#                     y, sr = librosa.load(file_path, sr=sr)
#                     sample_rates.add(sr)

#                     # Peak amplitude (dB)
#                     peak_amp = np.max(np.abs(y))
#                     peak_db = 20 * np.log10(peak_amp + 1e-10)

#                     # Spectral centroid
#                     centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
#                     mean_centroid = np.mean(centroid)

#                     # Find non-silent intervals
#                     non_silent = librosa.effects.split(y, top_db=top_db)
#                     total_duration = librosa.get_duration(y=y, sr=sr)

#                     # CASE A: Fully silent
#                     if len(non_silent) == 0:
#                         max_silence = total_duration
#                         silence_type = ["Fully Silent"]
#                     else:
#                         silence_durations = []
#                         silence_type = []

#                         # CASE B: Start silence
#                         if non_silent[0][0] > 0:
#                             silence_durations.append(non_silent[0][0] / sr)
#                             silence_type.append("Start")

#                         # CASE D: Middle silence
#                         for i in range(1, len(non_silent)):
#                             gap = (non_silent[i][0] - non_silent[i-1][1]) / sr
#                             if gap > 0:
#                                 silence_durations.append(gap)
#                                 silence_type.append("Middle")

#                         # CASE C: End silence
#                         if non_silent[-1][1] < len(y):
#                             silence_durations.append((len(y) - non_silent[-1][1]) / sr)
#                             silence_type.append("End")

#                         max_silence = max(silence_durations) if silence_durations else 0

#                     # Store if threshold exceeded
#                     if max_silence >= threshold_sec:
#                         records.append({
#                             "Genre": genre,
#                             "Stem": stem_name,
#                             "Duration": round(total_duration, 2),
#                             "Max_Silence_Sec": round(max_silence, 2),
#                             "Silence_Location": ", ".join(silence_type),
#                             "Peak Amplitudes": round(peak_db, 2),
#                             "Spectral Mean Centroid": round(mean_centroid, 2),
#                             "File_Path": file_path
#                         })

#                 except Exception as e:
#                     print(f"Error processing {file_path}: {e}")

#     print('sample_rates:', sample_rates)

#     df = pd.DataFrame(records)
#     return df

--- EXECUTION ---
Pass your 'tr' (training) dictionary here.
Ensure 'tr' is defined from your previous build_dataset code.

In [15]:

# df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

# #Total number of files with silence >= 5 seconds
# total_silence_files = len(df_silence) 
# print("Total files with silence >= 5s:", total_silence_files)

# #Filter only vocals from df_silence
# vocals_silence = df_silence[df_silence["Stem"] == "vocals"]

# #Count how many vocal tracks have silence >= 5 seconds
# total_vocals_silence = len(vocals_silence)

# print("Total vocal tracks with silence >= 5s:", total_vocals_silence)

# #Compute average silence length in seconds
# avg_silence_vocals = vocals_silence["Max_Silence_Sec"].mean()

# print("Average Silence Length in Vocals (secs):", round(avg_silence_vocals, 2))

# #Filter for jazz + drums
# jazz_drums_silence = df_silence[(df_silence["Genre"] == "jazz") & (df_silence["Stem"] == "drums")]

# #Count how many jazz drum tracks have silence >= 5 seconds
# total_jazz_drums_silence = len(jazz_drums_silence)

# print("Total jazz drum tracks with silence >= 5s:", total_jazz_drums_silence)

# #Filter for jazz + drums + silence location = only middle
# jazz_drums_middle_silence = df_silence[ (df_silence["Genre"] == "jazz") & (df_silence["Stem"] == "drums") & (df_silence["Silence_Location"] == "Middle") ]

# #Count how many tracks meet the condition
# total_jazz_drums_middle_silence = len(jazz_drums_middle_silence)

# print("Total jazz drum tracks with silence >= 5s and only middle:", total_jazz_drums_middle_silence)

# #Filter for jazz + drums + silence >= 5s + max silence >= 10s
# jazz_drums_long_silence = df_silence[ (df_silence["Genre"] == "jazz") & (df_silence["Stem"] == "drums") & (df_silence["Max_Silence_Sec"] >= 10) ]

# #Count how many tracks meet the condition
# total_jazz_drums_long_silence = len(jazz_drums_long_silence)

# print("Total jazz drum tracks with silence >= 5s and Max_Silence_Sec >= 10s:", total_jazz_drums_long_silence)

# average_peak_db = np.mean(df_silence['Peak Amplitudes'])

# print("Average Peak Amplitude (dB):", average_peak_db)

In [16]:
# # Mean spectral centroid across all blues files
# mean_spectral_centroid = np.mean(df_silence[df_silence["Genre"] == "blue"]['Spectral Mean Centroid'])
    
# print("Mean Spectral Centroid (Hz) for Blues genre:", mean_spectral_centroid)

# # Find genre with highest mean spectral centroid
# highest_genre = max(genre_centroids, key=genre_centroids.get)
# highest_value = genre_centroids[highest_genre]

# print("Genre with highest mean spectral centroid:", highest_genre)
# print("Mean Spectral Centroid (Hz):", highest_value)

In [17]:
# # --- RESULTS ANALYSIS ---

# # ------------------- write your code here -------------------------------
# #-------------------------------------------------------------------------
# # Hint: Create a pivot Table: Count by Genre vs Stem
# pivot = df_silence.pivot_table(index="Genre", columns="Stem", values="File_Path", aggfunc="count", fill_value=0)
# print(pivot)


In [18]:
# import librosa
# import numpy as np

# # Select the first song from 'rock'
# genre = "rock"
# song_index = 0
# song_path = os.path.join(DATA_ROOT, genre, os.listdir(os.path.join(DATA_ROOT, genre))[song_index])

# # Load all stems
# stems = []
# for stem_key in STEM_KEYS:
#     file_path = os.path.join(song_path, STEMS[stem_key])
#     y, sr = librosa.load(file_path, sr=None)
#     stems.append(y)

# # Align stems by length (pad/truncate to same size)
# min_len = min(len(s) for s in stems)
# stems = [s[:min_len] for s in stems]

# # Combine stems into a mix
# mix = np.sum(stems, axis=0)

# # Get duration of the mix sample
# mix_duration = librosa.get_duration(y=mix, sr=sr)
# print("Length of mix sample (secs):", round(mix_duration, 2))

# # Assuming you already created `mix` and have `sr` from the previous step
# # Compute RMS amplitude
# rms = librosa.feature.rms(y=mix)

# # Average RMS across frames
# rms_value = np.mean(rms)

# print("RMS Amplitude of mix sample:", rms_value)

# import numpy as np

# # Assuming `mix` is your combined audio signal
# # Peak normalization
# normalized_mix = mix / np.max(np.abs(mix))

# # Find the maximum value of the normalized sample
# max_peak_value = np.max(normalized_mix)

# print("Max value of peak normalized sample:", max_peak_value)


In [19]:
# stems_audio = []
# try:
#     for key in STEM_KEYS:
#       pass
#     # ------------------- write your code here -------------------------------
#     # Load audio (Duration 5.0s for speed/consistency)
#     #-------------------------------------------------------------------------
#     file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]

#     # Load audio (limit duration to 5.0s for speed/consistency)
#     y, sr = librosa.load(file_path, sr=None)
#     stems_audio.append((key, y, sr))

#     print("Audio loaded successfully.")
# except NameError:
#     print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
# except IndexError:
#     print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
# except Exception as e:
#     print(f"ERROR: {e}")

# mix_duration = librosa.get_duration(y=y, sr=sr) 
# print("Length of mix sample (secs):", round(mix_duration, 2))

In [20]:
# # ------------------- write your code here -------------------------------
# # Stack them into a numpy array (Shape: 4 x Samples)
# stems_stack = np.vstack([y for (_, y, _) in stems_audio])

# # Mix the stems by summing them element-wise
# mix_raw = np.sum(stems_stack, axis=0)

# # Calculate RMS Amplitude MANUALLY
# rms_val = np.sqrt(np.mean(mix_raw**2))
# print(rms_val)

# #Peak Normalization
# max_val = np.max(np.abs(mix_raw))

# if max_val > 0:
#     mix_norm = mix_raw / max_val
# else:
#     mix_norm = mix_raw

# # VALIDATION
# assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
# #------------------------------------------------------------------------
# print(max_val)

In [21]:
# # Average RMS across frames
# Jazz_duration_average = np.mean(df_silence[df_silence['Genre']=='jazz'].Duration)
# print(Jazz_duration_average)
# print(df_silence.head())